# dependecies



In [1]:
#clone git
!git clone https://github.com/danianamir/Tile-Based-Level-Design.git
%cd /content/Tile-Based-Level-Design
!git pull origin main

Cloning into 'Tile-Based-Level-Design'...
remote: Enumerating objects: 1439, done.
remote: Counting objects: 100% (213/213), done.
remote: Compressing objects: 100% (180/180), done.
remote: Total 1439 (delta 100), reused 32 (delta 8), pack-reused 1226
Receiving objects: 100% (1439/1439), 173.19 MiB | 10.24 MiB/s, done.
Resolving deltas: 100% (823/823), done.
Updating files: 100% (623/623), done.
/content/Tile-Based-Level-Design
From https://github.com/danianamir/Tile-Based-Level-Design
 * branch            main       -> FETCH_HEAD
Already up to date.


In [2]:
#package installs with pip_github......................................................................
%cd /content/Tile-Based-Level-Design/ml-agents-envs
!pip install .
!pip install gymnasium
!pip install ray[rllib]


/content/Tile-Based-Level-Design/ml-agents-envs
Processing /content/Tile-Based-Level-Design/ml-agents-envs
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.7/756.7 kB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for mlagents-envs: filename=mlagents_envs-0.30.0-py3-none-any.whl size=88754 sha256=c41f3bf50658307807eb123a754dbf5f6c039308802e1df7268af250570fb21e
  Stored in directory: /root/.cache/pip/wheels/2b/56/d7/23048fc8cc5dd066914bdd88bb14ab6ea85ce21552c480a402
  Created wheel for pettingzoo: filename=PettingZoo-1.15.0-py3-none-any.whl size=875631 sha256=8bdd017fe8b7f3a1d18166aae175ed9ac998e61c5f2d1b37f11f0e97b2250133
  Stored in directory: /root/.cache/pip/wheels/e3/35/ac/76984cb1c12902d190c818d57c43d25c3f9281591a640ccd13
Successfully built mlagents-envs pettingzoo
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 MB 9.0 M

In [3]:
import mlagents_envs
from mlagents_envs.environment import UnityEnvironment
from mlagents_envs.base_env import ActionTuple


#gymnasium
import gymnasium as gym
from gymnasium import spaces

#ray
import ray
from ray import  tune ,train
from ray.rllib.examples.policy.random_policy import RandomPolicy
from ray.rllib.evaluation.rollout_worker import RolloutWorker
from ray.rllib.evaluation.worker_set import WorkerSet
from ray.rllib.env.multi_agent_env import MultiAgentEnv
from ray.rllib.algorithms.algorithm_config import AlgorithmConfig
from ray.rllib.algorithms.algorithm import Algorithm
from ray.tune.registry import register_env
from ray.rllib.policy.policy import PolicySpec
from ray.rllib.env.multi_agent_env import MultiAgentEnv
from ray.rllib.algorithms.ppo.ppo import PPOConfig
from ray.rllib.algorithms.ppo.ppo import PPO
from ray.rllib.models import ModelCatalog
from ray.rllib.models.torch.torch_modelv2 import TorchModelV2
from ray.rllib.models import MODEL_DEFAULTS
from ray.rllib.policy.rnn_sequencing import add_time_dimension
from ray.rllib.models.torch.misc import SlimFC , SlimConv2d,normc_initializer

from ray.rllib.models.torch.fcnet import  FullyConnectedNetwork
from ray.rllib.models.torch.visionnet import  VisionNetwork
from ray.rllib.models.torch.recurrent_net import LSTMWrapper

from ray.rllib.policy.policy import Policy

# torch
import torch
from torch import nn
import torch.onnx as onnx



#others
import numpy as np
import random
import time
from pprint import pprint
from typing import Any, Dict, List, Optional, Tuple, Union
import os
import time
from ray.rllib.utils.annotations import override


/usr/local/lib/python3.10/dist-packages/flax/configurations.py:42: DeprecationWarning: jax.config.define_bool_state is deprecated. Please use other libraries for configuration instead.
  return jax_config.define_bool_state('flax_' + name, default, help)
/usr/local/lib/python3.10/dist-packages/flax/linen/activation.py:36: DeprecationWarning: jax.nn.normalize is deprecated. Use jax.nn.standardize instead.
  from jax.nn import normalize
/usr/local/lib/python3.10/dist-packages/tensorflow_probability/python/__init__.py:57: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if (distutils.version.LooseVersion(tf.__version__) <


In [4]:
# give access to run the linux file ................................................................................
%cd
!chmod -R 755 /content/Tile-Based-Level-Design/unity_built_linux/my_game.x86_64
!chmod -R 755 /content/Tile-Based-Level-Design/unity_built_linux/UnityPlayer.so
!ls -l /content/Tile-Based-Level-Design/unity_built_linux



/root


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


total 116244
drwxr-xr-x 3 root root     4096 Jan 29 16:25 my_game_BurstDebugInformation_DoNotShip
drwxr-xr-x 6 root root     4096 Jan 29 16:25 my_game_Data
-rw-r--r-- 1 root root    16040 Jan 29 16:25 my_game_s.debug
-rwxr-xr-x 1 root root    15008 Jan 29 16:25 my_game.x86_64
-rw-r--r-- 1 root root 69732336 Jan 29 16:25 UnityPlayer_s.debug
-rwxr-xr-x 1 root root 50427088 Jan 29 16:25 UnityPlayer.so


In [5]:
filename_linux="/content/Tile-Based-Level-Design/unity_built_linux/my_game.x86_64"
filename_algo ="/content/Tile-Based-Level-Design/ALGO/DISCRETE_local/neg"


# environment

In [6]:
class Unity3DEnv(MultiAgentEnv):


    _BASE_PORT_EDITOR = 5004
    _BASE_PORT_ENVIRONMENT = 5005
    _WORKER_ID = 0

    def __init__(
        self,
        file_name: str = None,
        port: Optional[int] = None,
        seed: int = 0,
        no_graphics: bool = False,
        timeout_wait: int = 300,
        episode_horizon: int = None,

    ):

        self._skip_env_checking = True
        super().__init__()

        if file_name is None:
            print(
                "No game binary provided, will use a running Unity editor "
                "instead.\nMake sure you are pressing the Play (|>) button in "
                "your editor to start."
            )

        # Try connecting to the Unity3D game instance. If a port is blocked
        port_ = None
        while True:
            # Sleep for random time to allow for concurrent startup of many
            # environments (num_workers >> 1). Otherwise, would lead to port
            # conflicts sometimes.
            if port_ is not None:
                time.sleep(random.randint(1, 10))
            port_ = port or (
                self._BASE_PORT_ENVIRONMENT if file_name else self._BASE_PORT_EDITOR
            )
            # cache the worker_id and
            # increase it for the next environment
            worker_id_ = Unity3DEnv._WORKER_ID if file_name else 0
            Unity3DEnv._WORKER_ID += 1
            try:
                self.unity_env = UnityEnvironment(
                    file_name=file_name,
                    worker_id=worker_id_,
                    base_port=port_,
                    seed=seed,
                    no_graphics=no_graphics,
                    timeout_wait=timeout_wait,
                )

                print("Created UnityEnvironment for port {}".format(port_ + worker_id_))
            except mlagents_envs.exception.UnityWorkerInUseException:
                pass
            else:
                break

        self.episode_horizon = episode_horizon
        self.episode_timesteps = 0






    def step(
        self, action_dict
    ):

        # it take a action_dict {"instantiator Behaviour" : tuple(spaces.box(shape(2,)) , multidiscrete([3,4]))  , "modifyer Behavior":tuple(box(shape(2,)) , spaces.multidiscrete([6,2]))}
        # make a list called actions for each beahivor for example
        # for "instantiator Behaviour" make : action[0] = tuple(spaces.box(shape(2,)) , spaces.discrete[3])
        # for "modifyer Behavior" make : action[0] = tuple(box(shape(2,)) , spaces.multidiscrete([6,2])

        for behavior_name in self.unity_env.behavior_specs:
                actions = []
                for agent_id in self.unity_env.get_steps(behavior_name)[0].agent_id:
                    key = behavior_name + "_{}".format(agent_id)
                    actions.append(action_dict[key])



                # take the actions list of the current behavior  and make a action_tuple list for current behavior
                # action tuple is a object that contain both discrete and continious action :
                # for example for  "instantiator Behaviour" :
                #action_tuple.continous = spaces.box(shape(2,))
                #action_tuple.discret= = spaces.discrete(3)


                    #the behaviorname == actuall behaviorname in unity +team=0
                    # for action tuple we had to pass a numpy arrye with shape (<number of agents>, <action size>)
                    if behavior_name=="instantiator Behaviour?team=0":
                       x=actions[0]["element_1"]
                       multi_discrete_action=np.array([x])
                       action_tuple =ActionTuple( discrete= multi_discrete_action)





                    # pass the current behavior and its action_tuple to step() function
                    self.unity_env.set_actions(behavior_name, action_tuple)

        self.unity_env.step()
        self.episode_timesteps += 1
        obs, rewards, terminateds, truncateds, infos = self._get_step_results()
        return obs, rewards, terminateds, truncateds, infos














    def reset(
        self, *, seed=None, options=None
    ):
        self.episode_timesteps = 0
        self.unity_env.reset()
        obs, _, _, _, infos = self._get_step_results()
        return obs, infos








    def _get_step_results(self):

        # first we set the initial dict that step return
        obs = {}
        rewards = {}
        terminated ={}
        truncated={}
        infos = {}


        num_active=0
        num_done=0
        num_all=0



        #go thorugh all the behavior
        # return decision_step (batch of agent have similar behavior)
        # return trminal steps  (batch of agent have similar behavior /that end thier episode)
        for behavior_name  in self.unity_env.behavior_specs:
            decision_steps, terminal_steps = self.unity_env.get_steps(behavior_name)
            num_active=num_active+len(decision_steps)
            num_done=num_done+len(terminal_steps)
            num_all=num_active+num_done





            #set the obs / reward  from  decision step for each agent like:{"behavior_name+agent_id" : numpy array}
            for agent_id, idx in decision_steps.agent_id_to_index.items():
                key = behavior_name + "_{}".format(agent_id)
                os = tuple(o[idx] for o in decision_steps.obs)
                os = os[0] if len(os) == 1 else os
                obs[key]=  os
                rewards[key] = (decision_steps.reward[idx] )



            #set the obs / reward  from  terminal_steps for each agent like:{"behavior_name+agent_id" : numpy array}
            for agent_id, idx in terminal_steps.agent_id_to_index.items():
                key = behavior_name + "_{}".format(agent_id)
                if key not in obs:
                    os = tuple(o[idx] for o in terminal_steps.obs)
                    os = os[0] if len(os) == 1 else os
                    obs[key]=  os

                rewards[key] = (terminal_steps.reward[idx] )


            #+ decision_steps.group_reward[idx]
         #+ terminal_steps.group_reward[idx]


        #infos={"acive":num_active , "done":num_done , "all":num_all }


        # set the terminated if all agent end thier own episod like: {"__all__":True}
        if(num_active>0):
           terminated["__all__"]=False
        else:
           terminated["__all__"]=True



        # set the truncated /if the step of the env reach the horizen like{"__all__":True}
        if self.episode_timesteps == self.episode_horizon:
          truncated["__all__"]=True
        else:
          truncated["__all__"]=False


        # return
        return obs, rewards, terminated, truncated, infos


# register env

In [7]:
# register the unity env
tune.register_env("unity3d",lambda _: Unity3DEnv(file_name=filename_linux,
                                                                                 episode_horizon=100,
                                                                                    no_graphics=True,
                                                                                )
                                                )

# model

In [8]:
class mine(  TorchModelV2, nn.Module):

    def __init__(self, obs_space, action_space, num_outputs, model_config, name):
        TorchModelV2.__init__( self, obs_space, action_space, num_outputs, model_config, name)
        nn.Module.__init__(self)

        self._features=None
        self.cell_size=model_config["lstm_cell_size"]



        #conv layer
        self.conv1 =SlimConv2d(
                   in_channels=1,
                    out_channels=32,
                    kernel=2,
                    stride=1,
                    padding=1,
                    activation_fn="relu",
                )






        #fc layer
        self.fc1=SlimFC(
            in_size=8,
            out_size=64,
            initializer=normc_initializer(1.0),
            activation_fn="relu",
        )



        # fc+conv concated
        self.fc2=SlimFC(
            in_size=576,
            out_size=512,
            initializer=normc_initializer(1.0),
            activation_fn="relu",
        )



        self.fc3=SlimFC(
            in_size=512,
            out_size=256,
            initializer=normc_initializer(1.0),
            activation_fn="relu",
        )




        #logit
        self.logit =   SlimFC(
            in_size=256,
            out_size=num_outputs,
            initializer=normc_initializer(0.01),
            activation_fn=None,
        )



        # value
        self.value=SlimFC(
            in_size=256,
            out_size=1,
            activation_fn=None,
           initializer=normc_initializer(0.01),
        )



    def forward(self, input_dict, state, seq_lens):

        self.conv_input=input_dict["obs_flat"][:,:9].unsqueeze(1).reshape(input_dict["obs_flat"].shape[0], 1, 3, 3)
        self.fc_input=  input_dict["obs_flat"][:,9:].float()


        #conv
        out=  self.conv1(self.conv_input)
        conv_out= out.view(out.size(0), -1)



        #fc
        fc_out=self.fc1(self.fc_input)




        #concat
        concat= torch.cat((  fc_out, conv_out), dim=1)
        concat_out=self.fc2(concat)

        #faeture
        self._features=self.fc3(concat_out)



        # logit
        output = self.logit(self._features)



        return output , state




    def value_function(self) :
        assert  self._features is not None, "must call forward() first"
        return torch.reshape(self.value( self._features), [-1])






In [9]:
model = mine(obs_space=spaces.Box(float("-inf"), float("inf"), (17,)),
                                            action_space= spaces.Dict({"element_1":spaces.MultiDiscrete([3,5])}),
                                            num_outputs=9,
                                            model_config=MODEL_DEFAULTS,
                                            name="my_model")


# initialize  input_dict  , seq_lent
obs = torch.rand(6,17)
#print(obs)
input_dict={"obs_flat":obs , "obs":obs }
seq_lens=np.array([0,])

#initialize state with  get_initial_state
state = model.get_initial_state()

# initialize sate with fake state
h0=torch.rand(1,256)
c0=torch.rand(1,256)
state=[h0,c0]


# foward
out = model(
            input_dict=input_dict,
            state=state,
            seq_lens=seq_lens
           )

#print("the output is {}".format(out))

# register model

In [10]:

ModelCatalog.register_custom_model("mine", mine)

# train the algorithm

In [11]:

import torch
print(torch.cuda.is_available())

False


In [12]:
discrete_instantiator = spaces.MultiDiscrete([3,5])





policies = { "policy_initializer":PolicySpec (
                                   observation_space= spaces.Box(float("-inf"), float("inf"), (17,)),
                                    action_space=spaces.Dict({"element_1":discrete_instantiator}),
                                    config={}
                                                                )


                }





def policy_mapping_fn(agent_id, episode, worker, **kwargs):
    if agent_id == 'instantiator Behaviour?team=0_0':
        return "policy_initializer"
    if agent_id =='modifyer Behavior?team=0_1':
        return "policy_modifier"





config=(
    PPOConfig()
    .environment("unity3d")

    .multi_agent(
                        policies=policies,
                         policy_mapping_fn=policy_mapping_fn
                        )

    .framework("torch")

    .rollouts(
            num_rollout_workers=2,

                )

    .resources(num_gpus=0)

    .training(
            lr=0.001,
            lambda_=0.95,
            gamma=0.99,
            sgd_minibatch_size=512,
            train_batch_size=10000,
            num_sgd_iter=30,
            clip_param=0.2,
            entropy_coeff=0,
            model={
                   "custom_model": "mine",
                    "custom_model_config": {},

                   },


            )
    .evaluation()
    .experimental(_enable_new_api_stack=False)
    .checkpointing(export_native_model_files=True)

    )

for key , item in config.items():
    print(key +":{}".format( item) )

extra_python_environs_for_driver:{}
extra_python_environs_for_worker:{}
num_gpus:0
num_cpus_per_worker:1
num_gpus_per_worker:0
_fake_gpus:False
num_learner_workers:0
num_gpus_per_learner_worker:0
num_cpus_per_learner_worker:1
local_gpu_idx:0
custom_resources_per_worker:{}
placement_strategy:PACK
eager_tracing:True
eager_max_retraces:20
tf_session_args:{'intra_op_parallelism_threads': 2, 'inter_op_parallelism_threads': 2, 'gpu_options': {'allow_growth': True}, 'log_device_placement': False, 'device_count': {'CPU': 1}, 'allow_soft_placement': True}
local_tf_session_args:{'intra_op_parallelism_threads': 8, 'inter_op_parallelism_threads': 8}
torch_compile_learner:False
torch_compile_learner_what_to_compile:forward_train
torch_compile_learner_dynamo_backend:inductor
torch_compile_learner_dynamo_mode:None
torch_compile_worker:False
torch_compile_worker_dynamo_backend:onnxrt
torch_compile_worker_dynamo_mode:None
env:unity3d
env_config:{}
observation_space:None
action_space:None
env_task_fn:No

In [13]:
algo=config.build()

/usr/local/lib/python3.10/dist-packages/ray/rllib/algorithms/algorithm.py:483: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
`UnifiedLogger` will be removed in Ray 2.7.
  return UnifiedLogger(config, logdir, loggers=None)
/usr/local/lib/python3.10/dist-packages/ray/tune/logger/unified.py:53: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
The `JsonLogger interface is deprecated in favor of the `ray.tune.json.JsonLoggerCallback` interface and will be removed in Ray 2.7.
  self._loggers.append(cls(self.config, self.logdir, self.trial))
/usr/local/lib/python3.10/dist-packages/ray/tune/logger/unified.py:53: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppre

(RolloutWorker pid=903) [UnityMemory] Configuration Parameters - Can be set up in boot.config
(RolloutWorker pid=903)     "memorysetup-bucket-allocator-granularity=16"
(RolloutWorker pid=903)     "memorysetup-bucket-allocator-bucket-count=8"
(RolloutWorker pid=903)     "memorysetup-bucket-allocator-block-size=4194304"
(RolloutWorker pid=903)     "memorysetup-bucket-allocator-block-count=1"
(RolloutWorker pid=903)     "memorysetup-main-allocator-block-size=16777216"
(RolloutWorker pid=903)     "memorysetup-thread-allocator-block-size=16777216"
(RolloutWorker pid=903)     "memorysetup-gfx-main-allocator-block-size=16777216"
(RolloutWorker pid=903)     "memorysetup-gfx-thread-allocator-block-size=16777216"
(RolloutWorker pid=903)     "memorysetup-cache-allocator-block-size=4194304"
(RolloutWorker pid=903)     "memorysetup-typetree-allocator-block-size=2097152"
(RolloutWorker pid=903)     "memorysetup-profiler-bucket-allocator-granularity=16"
(RolloutWorker pid=903)     "memorysetup-profil

(RolloutWorker pid=903) error: XDG_RUNTIME_DIR not set in the environment.


(RolloutWorker pid=903) FMOD failed to initialize the output device.: "Error initializing output device. " (60)
(RolloutWorker pid=903) Forced to initialize FMOD to to the device driver's system output rate 48000, this may impact performance and/or give inconsistent experiences compared to selected sample rate 48000
(RolloutWorker pid=903) FMOD failed to initialize the output device.: "Error initializing output device. " (60)
(RolloutWorker pid=903) FMOD initialized on nosound output
(RolloutWorker pid=903) Begin MonoManager ReloadAssembly


(RolloutWorker pid=903) ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
(RolloutWorker pid=903) ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
(RolloutWorker pid=903) ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
(RolloutWorker pid=903) ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
(RolloutWorker pid=903) ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
(RolloutWorker pid=903) ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
(RolloutWorker pid=903) ALSA lib conf.c:5701:(snd_config_expand) Evaluate error: No such file or directory
(RolloutWorker pid=903) ALSA lib pcm.c:2664:(snd_pcm_open_noupdate) Unknown PCM default
(RolloutWorker pid=903) ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
(RolloutWorker pid=903) ALSA lib conf.c:5178:(_snd_conf

(RolloutWorker pid=903) - Loaded All Assemblies, in  0.228 seconds
(RolloutWorker pid=903) - Finished resetting the current domain, in  0.004 seconds
(RolloutWorker pid=903) ERROR: Shader Sprites/Default shader is not supported on this GPU (none of subshaders/fallbacks are suitable)
(RolloutWorker pid=903) ERROR: Shader Sprites/Mask shader is not supported on this GPU (none of subshaders/fallbacks are suitable)
(RolloutWorker pid=903) UnloadTime: 1.353947 ms
(RolloutWorker pid=903) Registered Communicator in Agent.
(RolloutWorker pid=903) UnityEngine.StackTraceUtility:ExtractStackTrace () (at /home/bokken/build/output/unity/unity/Runtime/Export/Scripting/StackTrace.cs:37)
(RolloutWorker pid=903) UnityEngine.DebugLogHandler:LogFormat (UnityEngine.LogType,UnityEngine.Object,string,object[])
(RolloutWorker pid=903) UnityEngine.Logger:Log (UnityEngine.LogType,object)
(RolloutWorker pid=903) UnityEngine.Debug:Log (object)
(RolloutWorker pid=903) Unity.MLAgents.Agent:Awake () (at ./ml-agents

(RolloutWorker pid=903)   unity_communicator_version = StrictVersion(unity_com_ver)
(pid=902) /usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:121: DeprecationWarning: pkg_resources is deprecated as an API
(pid=902)   warnings.warn("pkg_resources is deprecated as an API", DeprecationWarning)
(pid=902) /usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:2870: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
(pid=903) Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages [repeated 5x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/ray-logging.html#log-deduplication for more options.)
(pid=903)   declare_namespace(pkg) [repeated 5x across cluster]
(pid=902) /usr/

In [14]:
#%%capture
results=[]
for i in range(100):
            result = algo.train()
            results.append(result)
            #pprint(results[i])
            print("traning iteration number {} finished".format(i))
            print("the min reward for result{} is: {}".format(i,results[i]['episode_reward_min']))
            print("the mean reward for result{} is: {}".format(i,results[i]['episode_reward_mean']))
            print("the max reward for result{} is: {} ".format(i,results[i]['episode_reward_max']))


            print("\n\n\n")





            if (
                result['timesteps_total'] >= 10000000000
                or result["episode_reward_mean"] >= 10000000000
                or result["episodes_total"]>=100000000000
            ):
                break

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


(RolloutWorker pid=903) Internal: deleting an allocation that is older than its permitted lifetime of 4 frames (age = 5)
(RolloutWorker pid=903) Internal: deleting an allocation that is older than its permitted lifetime of 4 frames (age = 5)
(RolloutWorker pid=903) Internal: deleting an allocation that is older than its permitted lifetime of 4 frames (age = 6)
(RolloutWorker pid=903) Internal: deleting an allocation that is older than its permitted lifetime of 4 frames (age = 7)
(RolloutWorker pid=903) Internal: deleting an allocation that is older than its permitted lifetime of 4 frames (age = 5)
(RolloutWorker pid=903) Internal: deleting an allocation that is older than its permitted lifetime of 4 frames (age = 5)
(RolloutWorker pid=903) Internal: deleting an allocation that is older than its permitted lifetime of 4 frames (age = 9)
(RolloutWorker pid=903) Internal: deleting an allocation that is older than its permitted lifetime of 4 frames (age = 9)
(RolloutWorker pid=903) Internal

2024-01-29 16:30:23,973	WARNING deprecation.py:50 -- DeprecationWarning: `_get_slice_indices` has been deprecated. This will raise an error in the future!


Streaming output truncated to the last 5000 lines.
(RolloutWorker pid=903) 
(RolloutWorker pid=903) [./Runtime/Allocator/ThreadsafeLinearAllocator.cpp line 248]
(RolloutWorker pid=903) 
(RolloutWorker pid=903) Internal: deleting an allocation that is older than its permitted lifetime of 4 frames (age = 6) [repeated 18x across cluster]
(RolloutWorker pid=903) Internal: deleting an allocation that is older than its permitted lifetime of 4 frames (age = 11) [repeated 24x across cluster]
(RolloutWorker pid=903) Internal: deleting an allocation that is older than its permitted lifetime of 4 frames (age = 6) [repeated 8x across cluster]
(RolloutWorker pid=902) Internal: deleting an allocation that is older than its permitted lifetime of 4 frames (age = 10) [repeated 2x across cluster]
(RolloutWorker pid=902) Internal: deleting an allocation that is older than its permitted lifetime of 4 frames (age = 12) [repeated 10x across cluster]
(RolloutWorker pid=902) Internal: deleting an allocation t

# load and test algorithm

In [ ]:
def load_algo(path_to_checkpoint):
   algo = Algorithm.from_checkpoint(path_to_checkpoint)
   return algo

algo=load_algo(filename_algo)


In [17]:
env =Unity3DEnv(file_name=filename_linux ,
                              episode_horizon=100  ,
                                no_graphics=True,
               )

Created UnityEnvironment for port 5007


/content/Tile-Based-Level-Design/ml-agents-envs/mlagents_envs/environment.py:94: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  unity_communicator_version = StrictVersion(unity_com_ver)


In [20]:
initial_state = algo.get_policy("policy_initializer").get_initial_state()
terminateds ={}
truncateds={}
num_win=0
num_loss=0


for i in range(20):
    obs,info=env.reset()
    terminateds['__all__'] = truncateds['__all__'] = False
    episode_reward=0
    timestep=0


    while not  terminateds['__all__'] and not  truncateds['__all__'] :
        action= algo.compute_single_action(observation=obs['instantiator Behaviour?team=0_0'] , state=initial_state , policy_id= "policy_initializer")
        action_dict={"instantiator Behaviour?team=0_0":action}
        obs, rewards, terminateds, truncateds, infos=env.step(action_dict)
        #print(rewards)
        episode_reward += rewards["instantiator Behaviour?team=0_0"]
        timestep +=1
    print("total time step is {}".format(timestep))
    print("total episode_reward is {}".format(episode_reward))
    if  terminateds['__all__'] :
        print("reach the goal "+"\n")
        num_win=num_win+1

    if  truncateds['__all__']:
        print("not reach"+"\n")
        num_loss=num_loss+1

print("number of win is {}".format(num_win))
print("number of loss is {}".format(num_loss))




total time step is 28
total episode_reward is 3.0
reach the goal 

total time step is 23
total episode_reward is 3.0
reach the goal 

total time step is 19
total episode_reward is 5.0
reach the goal 

total time step is 13
total episode_reward is 3.0
reach the goal 

total time step is 22
total episode_reward is 4.0
reach the goal 

total time step is 14
total episode_reward is 2.0
reach the goal 

total time step is 12
total episode_reward is 5.0
reach the goal 

total time step is 16
total episode_reward is 3.0
reach the goal 

total time step is 100
total episode_reward is 2.0
not reach

total time step is 19
total episode_reward is 5.0
reach the goal 

total time step is 33
total episode_reward is 4.0
reach the goal 

total time step is 16
total episode_reward is 7.0
reach the goal 

total time step is 18
total episode_reward is 6.0
reach the goal 

total time step is 7
total episode_reward is 2.0
reach the goal 

total time step is 17
total episode_reward is 6.0
reach the goal 

t